# Conversational Core

Turn-taking, scope handling, clarification, and memory across a conversation.

**Rollup: 57/57 cases passed** across 8 required capabilities, est. cost $0.1224, 6328s total.

This is an executive-level summary over already-captured real-LLM results -- see `docs/CAPABILITY_MAPPING.md` for the full 25-capability table, and `notebooks/capabilities/<NN>_<slug>/demo.ipynb` for every case in full detail per capability.

In [ ]:
import sys, pathlib

# Robust path insert regardless of where Jupyter's cwd lands (repo root, or
# this notebook's own folder under notebooks/capabilities/<slug>/):
_p = pathlib.Path.cwd()
while not (_p / "src").exists() and _p != _p.parent:
    _p = _p.parent
sys.path.insert(0, str(_p))

import os

try:
    from dotenv import load_dotenv  # optional: picks up a .env file if python-dotenv is installed
    load_dotenv(override=False)
except ImportError:
    pass

from src.orchestrator import Orchestrator
from src.llm_client import get_llm_client, GLOBAL_USAGE, MockLLMClient

provider = os.environ.get("LLM_PROVIDER", "").lower() or ("anthropic" if os.environ.get("ANTHROPIC_API_KEY") else "openai" if os.environ.get("OPENAI_API_KEY") else "mock")
print(f"LLM provider in use: {provider}" + ("  (\u26a0\ufe0f set ANTHROPIC_API_KEY or OPENAI_API_KEY for real answers)" if provider == "mock" else ""))

orch = Orchestrator()


LLM provider in use: openai


In [ ]:
def ask(question: str, label: str = ""):
    """Run one turn through the orchestrator and pretty-print everything the
    grader needs to see: routing, evidence sources, transparency notes, answer.
    Identical helper to notebooks/demo.ipynb -- see scripts/build_notebook.py."""
    if label:
        print(f"\n{'='*90}\n{label}\n{'='*90}")
    print(f"USER: {question}\n")
    resp = orch.handle_turn(question)
    print(f"[intent={resp.intent} | sub_agents={resp.sub_agents_used} | retried={resp.retried}]")
    if resp.citations:
        print(f"[citations: {[c['doc_id'] for c in resp.citations]}]")
    if resp.assumptions:
        print("[assumptions/limitations surfaced:]")
        for a in resp.assumptions:
            print(f"  - {a}")
    print(f"\nAGENT: {resp.answer}")
    if resp.follow_up_suggestions:
        print(f"\n(follow-up suggestions: {resp.follow_up_suggestions})")
    return resp


## 1. Single-turn and multi-turn conversational interactions

9/9 passed — $0.0265, 1526s

In [ ]:
_ = ask("What was North America's revenue in Q1 2024?", '01')
_ = ask("What was EMEA's EBITDA margin in Q2 2025?", '02')

01
USER: What was North America's revenue in Q1 2024?

[intent=data_query | sub_agents=['structured']]

AGENT: North America's revenue in **Q1 2024** was **$3,593 million (USD)** [structured KPI data].

Want to compare this with Q1 2025 or break it down by brand/segment?

02
USER: What was EMEA's EBITDA margin in Q2 2025?

[intent=data_query | sub_agents=['structured']]

AGENT: EMEA’s EBITDA margin in Q2 2025 was **32.1%**.

## 2. Greeting, capability introduction, and out-of-scope request handling

10/10 passed — $0.0051, 317s

In [ ]:
_ = ask('hi', '01')
_ = ask('Hello!', '02')

01
USER: hi

[intent=greeting | sub_agents=[]]

AGENT: Hello! I'm the Anheuser-Busch InBev (AB InBev) Q&A assistant. Anheuser-Busch InBev (AB InBev)'s real, publicly disclosed financial performance -- revenue, volume, normalized EBITDA, EBITDA margin, organic revenue growth and net profit -- by reporting zone (North America, Middle Americas, South America, EMEA, Asia Pacific) and by quarter or year, plus qualitative country- and brand-level commentary, company news, and competitive context drawn from AB InBev's own results releases and filings. Ask me about revenue, volume, EBITDA, margin, organic growth or net profit by zone/period, or about brand and country context from our documents. What would you like to know?

02
USER: Hello!

[intent=greeting | sub_agents=[]]

AGENT: Hello! I'm the Anheuser-Busch InBev (AB InBev) Q&A assistant. Anheuser-Busch InBev (AB InBev)'s real, publicly disclosed financial performance -- revenue, volume, normalized EBITDA, EBITDA margin, organic revenue g

## 3. Intent validation before data retrieval

8/8 passed — $0.0062, 318s

In [ ]:
_ = ask('hi', '01')
_ = ask("What's the weather?", '02')

01
USER: hi

[intent=greeting | sub_agents=[]]

AGENT: Hello! I'm the Anheuser-Busch InBev (AB InBev) Q&A assistant. Anheuser-Busch InBev (AB InBev)'s real, publicly disclosed financial performance -- revenue, volume, normalized EBITDA, EBITDA margin, organic revenue growth and net profit -- by reporting zone (North America, Middle Americas, South America, EMEA, Asia Pacific) and by quarter or year, plus qualitative country- and brand-level commentary, company news, and competitive context drawn from AB InBev's own results releases and filings. Ask me about revenue, volume, EBITDA, margin, organic growth or net profit by zone/period, or about brand and country context from our documents. What would you like to know?

02
USER: What's the weather?

[intent=out_of_scope | sub_agents=[]]

AGENT: That's outside what I can help with -- I'm scoped to Anheuser-Busch InBev (AB InBev)'s business data and related market/company context. Anheuser-Busch InBev (AB InBev)'s real, publicly disclosed f

## 4. Clarification for ambiguous or incomplete user requests

8/8 passed — $0.0067, 401s

In [ ]:
_ = ask('Tell me about performance.', '01')
_ = ask('How is it doing?', '02')

01
USER: Tell me about performance.

[intent=clarification_needed | sub_agents=[]]

AGENT: Which aspect of AB InBev's performance would you like? For example, a specific KPI (revenue, volume, normalized EBITDA, etc.), reporting zone, and time period (quarter/year)?

02
USER: How is it doing?

[intent=clarification_needed | sub_agents=[]]

AGENT: What would you like to know about? For example, AB InBev's overall performance, a specific zone, country, or brand?

## 5. Contextual follow-up questions by maintaining conversation history

7/7 passed — $0.0185, 939s

In [ ]:
_ = ask("What was North America's revenue in Q1 2024?", '01')
_ = ask("What was EMEA's revenue in Q3 2025?", '02')

01
USER: What was North America's revenue in Q1 2024?

[intent=data_query | sub_agents=['structured']]

AGENT: North America's revenue in **Q1 2024** was **$3,593 million (USD)** [structured KPI data].

Want to compare this with Q1 2025 or break it down by brand/segment?

02
USER: What was EMEA's revenue in Q3 2025?

[intent=data_query | sub_agents=['structured']]

AGENT: EMEA’s revenue in Q3 2025 was **$2,524 million** (USD million) per the structured KPI data.

Would you like to see how this compares quarter-over-quarter or year-over-year, or a breakdown by market/brand within EMEA?

## 8. Conversation context preservation across interactions

6/6 passed — $0.0196, 882s

In [ ]:
_ = ask("What was North America's revenue in Q1 2024?", '01')
_ = ask("Tell me about Stella Artois's positioning.", '02')

01
USER: What was North America's revenue in Q1 2024?

[intent=data_query | sub_agents=['structured']]

AGENT: North America's revenue in **Q1 2024** was **$3,593 million (USD)** [structured KPI data].

Want to compare this with Q1 2025 or break it down by brand/segment?

02
USER: Tell me about Stella Artois's positioning.

[intent=data_query | sub_agents=['unstructured']]
[citations: ['DOC-012', 'DOC-004', 'DOC-014', 'DOC-011', 'DOC-015']]

AGENT: Based on the retrieved evidence, **Stella Artois is positioned as one of AB InBev’s global “megabrands”** — the portfolio anchored by Budweiser, Corona, Stella Artois and Michelob Ultra [DOC-012]. This megabrand group grew revenue **4.6% in FY2024** and **4.1% in FY2025**, both ahead of total company revenue growth, suggesting Stella Artois sits within AB InBev’s global growth engine [DOC-012].

That said, the retrieved documents do **not** contain a detailed brand‑positioning statement for Stella Artois — e.g., target consumer, price tier, 

## 16. Context-aware follow-up suggestions within supported business domains

6/6 passed — $0.0072, 466s

In [ ]:
_ = ask("What was North America's revenue in Q1 2024?", '01')
_ = ask("What was EMEA's EBITDA margin in 2025?", '02')

01
USER: What was North America's revenue in Q1 2024?

[intent=data_query | sub_agents=['structured']]

AGENT: North America's revenue in **Q1 2024** was **$3,593 million (USD)** [structured KPI data].

Want to compare this with Q1 2025 or break it down by brand/segment?

02
USER: What was EMEA's EBITDA margin in 2025?

[intent=data_query | sub_agents=['structured']]

AGENT: EMEA’s EBITDA margin in 2025 was **32.6%**.

## 17. Conversation memory optimization for long-running sessions

3/3 passed — $0.0326, 1479s

In [ ]:
_ = ask('What was South America revenue in 2024?', '01')
_ = ask('What was South America revenue in 2024?', '02')

01
USER: What was South America revenue in 2024?

[intent=data_query | sub_agents=['structured']]

AGENT: South America revenue in 2024 was **$12,423 million USD** (~$12.4 billion), based on the quarterly KPI data.

Would you like a quarterly breakdown, or a comparison with 2023 revenue for South America?

02
USER: What was South America revenue in 2024?

[intent=data_query | sub_agents=['structured']]

AGENT: South America revenue in 2024 was **$12,423 million USD** (~$12.4 billion), based on the quarterly KPI data.

Would you like a quarterly breakdown, or a comparison with 2023 revenue for South America?